## Generate Synthetic Data with AI

In this lab, we use Azure OpenAI and a structured schema (defined using Pydantic) to generate realistic synthetic data. The goal is to automatically create test-ready records that match your expected JSON structure, using AI to populate fields with coherent, domain-specific values. This approach eliminates manual test data creation, ensures consistency, and helps validate your systems at scale with representative datasets.

## Step 1: Set Up Your Environment

In [ ]:
!pip install -qU openai pydantic --upgrade

These libraries are required for Azure OpenAI and schema validation.



## Step 2: Load Your Sample JSON

Create a file `sample.json` with content like:


In [ ]:
import json

with open('data/data.json', 'r') as f:
    data = json.load(f)


## Step 3: Define Schema Using Pydantic

In [ ]:
from pydantic import BaseModel
from typing import List

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip: str
    country: str

class Company(BaseModel):
    id: str
    name: str
    address: Address

class Employee(BaseModel):
    id: str
    name: str
    role: str
    hire_date: str
    salary: float
    active: bool

class InventoryItem(BaseModel):
    item_id: str
    name: str
    category: str
    quantity: int
    unit_price: float

class OrderLine(BaseModel):
    item_id: str
    quantity: int
    unit_price: float

class Order(BaseModel):
    order_id: str
    customer_id: str
    order_date: str
    status: str
    lines: List[OrderLine]
    total_amount: float

class ERPData(BaseModel):
    company: Company
    employees: List[Employee]
    inventory: List[InventoryItem]
    orders: List[Order]


This tells Azure to adhere to the structure.

## Step 4: Configure the Azure OpenAI Client (please get the Azure_open_AI_endpoint, Azure_open_Key, Model Deployment name from the instructor)

In [ ]:
import os
from openai import AzureOpenAI

os.environ["AZURE_OPENAI_API_KEY"] = "9i3uIwM1g5xLEY09P719g4F75604SDLZtPtGZ5fBiZeTF2t5Xv0EJQQJ99BLACYeBjFXJ3w3AAABACOGFqbK"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://agenticaiengineer.openai.azure.com/"

client = AzureOpenAI(
    api_key= os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint= os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    api_version="2025-01-01-preview",

)

## Step 5: Call the API for Structured JSON Output

In [ ]:
num = 5

prompt = f"""
You are a data generation assistant.

Your task is to generate a JSON object with a field called "records" that contains exactly {num} examples of synthetic ERP data.

Each item in the "records" array must strictly follow the structure and field types of the example provided below.

### Guidelines:
- All fields present in the sample must be included in each record.
- Do not leave any fields blank or null. Populate every field with realistic and coherent synthetic values.
- Maintain correct data types (e.g., strings, numbers, booleans, nested objects, and arrays).
- Use domain-appropriate values: realistic names, addresses, job roles, inventory items, etc.
- Ensure date formats match the sample.
- Preserve all nested structures and list fields exactly as in the sample.
- If the sample includes nested lists (e.g., orders → lines), generate multiple valid sub-items per record.
- Do not include explanatory text, comments, or markdown — only return a valid JSON object.

### Sample Record:
{json.dumps(data, indent=2)}

Return only a valid JSON object like:
{{
  "records": [ {{ ... }}, {{ ... }}, ..., {{ ... }} ]
}}
"""


In [ ]:
from pydantic import BaseModel
from typing import List

class ERPDataWrapper(BaseModel):
    records: List[ERPData]

response = client.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=ERPDataWrapper,
    messages=[
        {"role": "system", "content": "Generate a JSON object with a key 'records' whose value is a list of items."},
        {"role": "user", "content": prompt}
    ]
)

erp_list = response.choices[0].message.parsed

In [ ]:
for i, record in enumerate(erp_list.records, 1):
    print(f"\n--- ERP Record {i} ---")
    print(record.model_dump_json(indent=2))

Save the generated data.

In [ ]:
import json

erp_dict = erp_list.model_dump()

with open("synthetic_erp.json", "w") as f:
    json.dump(erp_dict, f, indent=2)

print("Saved to synthetic_erp.json")


## Why This Works

* **Structured Outputs** mode ensures model output strictly follows the schema.
* Pydantic checks both structure and types on the Python side .


## Recap for Your Lab

1. **Get sample JSON.**
2. **Define schema** in Pydantic.
3. **Install dependencies.**
4. **Authenticate and initialize** Azure OpenAI client.
5. **Prompt the model** to generate synthetic data matching your structure.
6. **Parse and validate** the model output.

This provides a reproducible, extensible framework for you to **automate synthetic test data generation**, ensuring consistency, validity, and easy scaling for your testing needs.



## Conclusion

In this lab, you successfully explored how to generate structured synthetic data using Azure OpenAI and Pydantic models. By combining schema-driven prompting with language model capabilities, you automated the creation of realistic, test-ready JSON data—ideal for ERP systems and beyond. This approach not only reduces manual effort in test data generation but also ensures schema compliance, data variety, and faster validation cycles. The modular setup allows you to scale across data formats, domains, and record volumes, making it a powerful tool for modern testing workflows.
